In [ ]:
# Matrix Factorization with BPR loss (MF-BPR)

In [13]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [14]:
# Загрузка данных
df = pd.read_parquet('df.parquet', engine='fastparquet')
client_df = pd.read_parquet('client_data.parquet', engine='fastparquet')
df_test = pd.read_parquet('df_test.parquet', engine='fastparquet')

TEST_MODE = False
SAMPLE_USERS = 500

if TEST_MODE:
    sample_users = np.random.choice(df['Телефон_new'].unique(), SAMPLE_USERS, replace=False)
    df = df[df['Телефон_new'].isin(sample_users)]
    client_df = client_df[client_df['Телефон_new'].isin(sample_users)]
    df_test = df_test[df_test['Телефон_new'].isin(sample_users)]

In [3]:
# Dataset для BPR
class BPRDataset(Dataset):
    def __init__(self, df, user_to_idx, item_to_idx, num_negatives=5):
        self.user_to_idx = user_to_idx
        self.item_to_idx = item_to_idx
        self.num_negatives = num_negatives
        
        self.user_items = {}
        for _, row in df.iterrows():
            u = user_to_idx[row['Телефон_new']]
            i = item_to_idx[row['ID_SKU']]
            if u not in self.user_items:
                self.user_items[u] = set()
            self.user_items[u].add(i)
        
        self.all_items = set(item_to_idx.values())
        self.positive_pairs = [(u, i) for u, items in self.user_items.items() for i in items]
    
    def __len__(self):
        return len(self.positive_pairs)
    
    def __getitem__(self, idx):
        u, i = self.positive_pairs[idx]
        neg_candidates = list(self.all_items - self.user_items[u])
        j = np.random.choice(neg_candidates)
        return torch.LongTensor([u]), torch.LongTensor([i]), torch.LongTensor([j])

In [4]:
# MF-BPR модель
class MFBPR(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=128):
        super(MFBPR, self).__init__()
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)
    
    def forward(self, user_idx, item_idx):
        user_vec = self.user_embedding(user_idx)
        item_vec = self.item_embedding(item_idx)
        return (user_vec * item_vec).sum(dim=1)
    
    def predict(self, user_idx, all_item_idx):
        user_vec = self.user_embedding(user_idx)
        item_vec = self.item_embedding(all_item_idx)
        return torch.mm(user_vec, item_vec.t()).squeeze()

In [5]:
# Функция для создания маппингов
def create_mappings(df_train):
    users = df_train['Телефон_new'].unique()
    items = df_train['ID_SKU'].unique()
    user_to_idx = {u: i for i, u in enumerate(users)}
    idx_to_user = {i: u for i, u in enumerate(users)}
    item_to_idx = {it: i for i, it in enumerate(items)}
    idx_to_item = {i: it for i, it in enumerate(items)}
    return user_to_idx, idx_to_user, item_to_idx, idx_to_item, len(users), len(items)

In [6]:
# Функция подбора гиперпараметров
def tune_mfbpr_hyperparams(df_train, val_users_data=None, train_val_split=0.1, 
                           embedding_dims=[32, 64, 128, 256],
                           learning_rates=[0.01, 0.05, 0.1],
                           regs=[0.001, 0.01, 0.1, 1.0],
                           batch_size=1024, random_state=42):
    
    if val_users_data is None:
        all_users = df_train['Телефон_new'].unique()
        train_users, val_users = train_test_split(all_users, test_size=train_val_split, 
                                                   random_state=random_state)
        df_train_subset = df_train[df_train['Телефон_new'].isin(train_users)]
        df_val = df_train[df_train['Телефон_new'].isin(val_users)]
        
        val_data = {}
        for user in val_users:
            user_transactions = df_val[df_val['Телефон_new'] == user].sort_values('Дата')
            if len(user_transactions) < 2:
                continue
            last_date = user_transactions['Дата'].max()
            last_order_items = user_transactions[user_transactions['Дата'] == last_date]['ID_SKU'].tolist()
            prev_items = user_transactions[user_transactions['Дата'] < last_date]['ID_SKU'].tolist()
            if len(prev_items) > 0 and len(last_order_items) > 0:
                val_data[user] = {'bought': prev_items, 'true_items': last_order_items}
    else:
        val_data = val_users_data
        df_train_subset = df_train
    
    user_to_idx, _, item_to_idx, _, n_users, n_items = create_mappings(df_train_subset)
    train_dataset = BPRDataset(df_train_subset, user_to_idx, item_to_idx, num_negatives=5)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    best_hr = 0
    best_params = {'dim': 128, 'lr': 0.05, 'reg': 0.01}
    
    for dim in embedding_dims:
        for lr in learning_rates:
            for reg in regs:
                model = MFBPR(n_users, n_items, embedding_dim=dim)
                optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=reg)
                model.to(device)
                
                # Быстрое обучение на 10 эпох
                for epoch in range(10):
                    model.train()
                    for u, i, j in train_loader:
                        u, i, j = u.to(device), i.to(device), j.to(device)
                        optimizer.zero_grad()
                        x_ui = model(u.squeeze(), i.squeeze())
                        x_uj = model(u.squeeze(), j.squeeze())
                        loss = -torch.log(torch.sigmoid(x_ui - x_uj)).mean()
                        loss.backward()
                        optimizer.step()
                
                # Оценка на валидации
                hits = 0
                all_item_idx = torch.arange(n_items, device=device)
                
                with torch.no_grad():
                    for user_id, data in val_data.items():
                        if user_id not in user_to_idx:
                            continue
                        user_idx = user_to_idx[user_id]
                        user_tensor = torch.LongTensor([user_idx]).to(device)
                        scores = model.predict(user_tensor, all_item_idx).cpu().numpy()
                        
                        bought_idx = [item_to_idx[item] for item in data['bought'] if item in item_to_idx]
                        scores[bought_idx] = -np.inf
                        
                        top_k_idx = np.argsort(scores)[-10:][::-1]
                        rec_items = [idx_to_item.get(idx, None) for idx in top_k_idx]
                        
                        if any(item in data['true_items'] for item in rec_items):
                            hits += 1
                
                hr = hits / len(val_data) if len(val_data) > 0 else 0
                
                if hr > best_hr:
                    best_hr = hr
                    best_params = {'dim': dim, 'lr': lr, 'reg': reg}
    
    return best_params, best_hr

In [7]:
# Функция обучения модели с заданными параметрами
def train_mfbpr_model(df_train, embedding_dim=128, lr=0.05, reg=0.01, epochs=30, 
                      batch_size=1024, tune=True, param_grid=None):
    
    if tune:
        param_grid = param_grid or {}
        best_params, _ = tune_mfbpr_hyperparams(df_train, **param_grid)
        embedding_dim = best_params['dim']
        lr = best_params['lr']
        reg = best_params['reg']
    
    user_to_idx, idx_to_user, item_to_idx, idx_to_item, n_users, n_items = create_mappings(df_train)
    
    dataset = BPRDataset(df_train, user_to_idx, item_to_idx, num_negatives=5)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = MFBPR(n_users, n_items, embedding_dim=embedding_dim)
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=reg)
    model.to(device)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for u, i, j in loader:
            u, i, j = u.to(device), i.to(device), j.to(device)
            optimizer.zero_grad()
            x_ui = model(u.squeeze(), i.squeeze())
            x_uj = model(u.squeeze(), j.squeeze())
            loss = -torch.log(torch.sigmoid(x_ui - x_uj)).mean()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    
    model_dict = {
        'model': model,
        'user_to_idx': user_to_idx,
        'idx_to_user': idx_to_user,
        'item_to_idx': item_to_idx,
        'idx_to_item': idx_to_item,
        'embedding_dim': embedding_dim,
        'lr': lr,
        'reg': reg,
        'df_train': df_train
    }
    
    return model_dict

In [8]:
# Функция рекомендаций
def recommend_mfbpr(user_id, model_dict, n_recommendations=10):
    if user_id not in model_dict['user_to_idx']:
        popular = model_dict['df_train']['ID_SKU'].value_counts().head(n_recommendations).index.tolist()
        return popular
    
    user_idx = model_dict['user_to_idx'][user_id]
    user_tensor = torch.LongTensor([user_idx]).to(device)
    n_items = len(model_dict['item_to_idx'])
    all_item_idx = torch.arange(n_items, device=device)
    
    with torch.no_grad():
        scores = model_dict['model'].predict(user_tensor, all_item_idx).cpu().numpy()
    
    bought = set(model_dict['df_train'][model_dict['df_train']['Телефон_new'] == user_id]['ID_SKU'].unique())
    bought_idx = [model_dict['item_to_idx'][item] for item in bought if item in model_dict['item_to_idx']]
    scores[bought_idx] = -np.inf
    
    top_k_idx = np.argsort(scores)[-n_recommendations:][::-1]
    recommendations = [model_dict['idx_to_item'][idx] for idx in top_k_idx if idx in model_dict['idx_to_item']]
    
    return recommendations

In [9]:
# Функция оценки
def evaluate_mfbpr(test_grouped, model_dict, k=10):
    hits = 0
    map_sum = 0.0
    
    for _, row in test_grouped.iterrows():
        user = row['Телефон_new']
        true_items = row['true_items']
        
        recs = recommend_mfbpr(user, model_dict, n_recommendations=k)
        
        hits_in_recs = [item for item in true_items if item in recs]
        if len(hits_in_recs) > 0:
            hits += 1
            positions = [recs.index(item) + 1 for item in hits_in_recs]
            map_sum += np.mean([1.0 / p for p in positions])
    
    return {'HitRate@K': hits / len(test_grouped), 'MAP@K': map_sum / len(test_grouped)}

In [10]:
# Подготовка тестовых данных
test_grouped = df_test.groupby('Телефон_new').agg(
    true_items=('ID_SKU', list)
).reset_index()
test_grouped = test_grouped.merge(
    client_df[['Телефон_new', 'cluster_5']], 
    on='Телефон_new', 
    how='inner'
)

In [15]:
# # Глобальная модель с подбором гиперпараметров
global_model = train_mfbpr_model(df, tune=True)
print(f"Global model: dim={global_model['embedding_dim']}, lr={global_model['lr']}, reg={global_model['reg']}")

for k in [5, 10, 20]:
    m = evaluate_mfbpr(test_grouped, global_model, k=k)
    print(f"K={k}: HR={m['HitRate@K']:.4f}, MAP={m['MAP@K']:.4f}")


Global model: dim=128, lr=0.05, reg=0.01
K=5: HR=0.1040, MAP=0.0470
K=10: HR=0.1610, MAP=0.0660
K=20: HR=0.2260, MAP=0.0810


In [ ]:
# Сегментированные модели (по кластерам)
cluster_models = {}

for cluster_id in sorted(df['cluster_5'].unique()):
    users_in_cluster = client_df[client_df['cluster_5'] == cluster_id]['Телефон_new'].unique()
    df_cluster = df[df['Телефон_new'].isin(users_in_cluster)]
    
    if len(df_cluster) < 100:
        continue
    
    try:
        cluster_model = train_mfbpr_model(df_cluster, tune=True, param_grid={
            'embedding_dims': [32, 64, 128],
            'learning_rates': [0.01, 0.05, 0.1],
            'regs': [0.001, 0.01, 0.1]
        })
        cluster_models[cluster_id] = cluster_model
    except Exception as e:
        continue

In [18]:
# Оценка сегментированных моделей
def recommend_segmented(user_id, n_recommendations=10):
    user_row = client_df[client_df['Телефон_new'] == user_id]
    if len(user_row) == 0:
        return recommend_mfbpr(user_id, global_model, n_recommendations)
    
    cluster_id = user_row.iloc[0]['cluster_5']
    
    if cluster_id not in cluster_models:
        return recommend_mfbpr(user_id, global_model, n_recommendations)
    
    if user_id not in cluster_models[cluster_id]['user_to_idx']:
        return recommend_mfbpr(user_id, global_model, n_recommendations)
    
    return recommend_mfbpr(user_id, cluster_models[cluster_id], n_recommendations)

for c in sorted(test_grouped['cluster_5'].unique()):
    ct = test_grouped[test_grouped['cluster_5'] == c]
    if len(ct) == 0:
        continue
    
    hits = 0
    map_sum = 0.0
    for _, row in ct.iterrows():
        recs = recommend_segmented(row['Телефон_new'], 10)
        hits_in_recs = [item for item in row['true_items'] if item in recs]
        if len(hits_in_recs) > 0:
            hits += 1
            positions = [recs.index(item) + 1 for item in hits_in_recs]
            map_sum += np.mean([1.0 / p for p in positions])
    
    hr = hits / len(ct)
    map_k = map_sum / len(ct)
    print(f"Cluster {c}: n={len(ct)}, HR@10={hr:.4f}, MAP@10={map_k:.4f}")


Cluster 0: n=17368, HR@10=0.0670, MAP@10=0.0270
Cluster 1: n=12768, HR@10=0.2210, MAP@10=0.0830
Cluster 2: n=19387, HR@10=0.1710, MAP@10=0.0630
Cluster 3: n=5764, HR@10=0.1260, MAP@10=0.0460
Cluster 4: n=25508, HR@10=0.1390, MAP@10=0.0530



In [19]:
# Сохранение результатов
results = test_grouped[['Телефон_new', 'cluster_5']].copy()

hits_list, maps_list = [], []
for _, row in test_grouped.iterrows():
    recs = recommend_mfbpr(row['Телефон_new'], global_model, 10)
    hits_in_recs = [item for item in row['true_items'] if item in recs]
    if len(hits_in_recs) > 0:
        hits_list.append(1)
        positions = [recs.index(item) + 1 for item in hits_in_recs]
        maps_list.append(np.mean([1.0 / p for p in positions]))
    else:
        hits_list.append(0)
        maps_list.append(0.0)

results['mfbpr_hit'] = hits_list
results['mfbpr_map'] = maps_list
results.to_parquet('results_mfbpr.parquet', engine='fastparquet', index=False)